# LLM 03: Tokenization

Hands-on:
1. Train a tiny BPE from scratch
2. Use `tiktoken` to tokenize with real OpenAI encodings
3. Compare token counts across languages, code, JSON
4. Apply Llama / Claude chat templates via HuggingFace
5. Exercises: token-budget estimator, anomalous tokens

Deps: `pip install tiktoken transformers`

## 1. Tiny BPE from Scratch

Merge the most-frequent byte pair repeatedly until vocab reaches target size.

In [ ]:
from collections import Counter

def get_stats(tokens):
    return Counter(zip(tokens, tokens[1:]))

def merge(tokens, pair, new_id):
    out = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(tokens[i])
            i += 1
    return out

text = "the quick brown fox jumps over the lazy dog the quick brown fox" * 50
tokens = list(text.encode('utf-8'))   # start from raw bytes
merges = {}
vocab_size = 270   # 256 bytes + 14 new merges
next_id = 256

while next_id < vocab_size:
    stats = get_stats(tokens)
    if not stats:
        break
    top_pair = max(stats, key=stats.get)
    tokens = merge(tokens, top_pair, next_id)
    merges[top_pair] = next_id
    next_id += 1

print(f'vocab size: {next_id}')
print(f'learned {len(merges)} merges')
print(f'compressed from {len(text.encode())} bytes to {len(tokens)} tokens '
      f'(ratio {len(text.encode())/len(tokens):.2f}x)')

## 2. Real Tokenizers via `tiktoken`

In [ ]:
import tiktoken

# OpenAI encodings
enc_gpt4 = tiktoken.encoding_for_model('gpt-4')
enc_gpt4o = tiktoken.encoding_for_model('gpt-4o')

samples = {
    'English':      'The quick brown fox jumps over the lazy dog.',
    'Rare word':    'Antidisestablishmentarianism',
    'Japanese':     'こんにちは、世界',
    'Code':         'def add(a, b): return a + b',
    'JSON':         '{"name": "Alice", "age": 30, "active": true}',
    'Number':       '3141592653589793',
}

print(f"{'sample':<12} {'gpt-4':>7} {'gpt-4o':>8}")
for name, text in samples.items():
    print(f'{name:<12} {len(enc_gpt4.encode(text)):>7} {len(enc_gpt4o.encode(text)):>8}')

## 3. Peek Inside: What Do the Tokens Look Like?

In [ ]:
text = 'The quick brown fox'
ids = enc_gpt4o.encode(text)
pieces = [enc_gpt4o.decode([i]) for i in ids]
for i, p in zip(ids, pieces):
    print(f'{i:>6}  "{p}"')

## 4. Llama-3 Chat Template via HuggingFace

Never hand-roll chat templates. Use the tokenizer that ships with the model.

In [ ]:
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B-Instruct')
    messages = [
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user',   'content': 'What is the capital of France?'},
    ]
    templated = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    print(templated)
    print('\ntoken count:', len(tok.encode(templated)))
except Exception as e:
    print('Skipped (need HF login + model access):', e)
    print('Sketch of what the template looks like:')
    print('<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
          'You are a helpful assistant.<|eot_id|>'
          '<|start_header_id|>user<|end_header_id|>\n\n'
          'What is the capital of France?<|eot_id|>'
          '<|start_header_id|>assistant<|end_header_id|>\n\n')

## 5. Exercise: Build a Token-Cost Estimator

Write `estimate_cost(prompt: str, model: str, completion_tokens: int) -> float` that:

1. Counts input tokens for the given model via tiktoken.
2. Looks up `(input_price_per_M, output_price_per_M)` from a dict of 2026 pricing.
3. Returns the total cost.

Then test with 1000 representative prompts sampled from your actual use case; report p50 / p95 / p99 cost. This is the seed of every serious feature-level cost model.

## 6. Exercise: Hunt Anomalous Tokens

Some training artifacts persist as lone vocabulary entries that the model never learned to handle well (the famous "SolidGoldMagikarp" phenomenon).

**Try it:** iterate over every token ID in `tiktoken`'s GPT-4 encoding, decode it, and flag ones that look like scraped usernames, nonsense, or repeated fragments. Send a few of the weirdest ones to an actual model and observe the response.

## 7. Takeaways

- **Subword BPE is standard.** Characters too long; words too sparse.
- **Token counts differ across models.** Measure, don't estimate.
- **Chat templates are model-specific.** Always use the bundled tokenizer.
- **Non-English and code can cost 2–3x more tokens.** Plan your unit economics.
- **Ship tokenizer + model together** for any fine-tune or self-hosted deployment.

Next: **04 — Embeddings**, where token IDs become the dense vectors the model actually computes with.